In [3]:
import pandas as pd
from pathlib import Path

downloads = Path.home() / "Downloads"

In [5]:
general = pd.read_csv(downloads / "FlightInfo_general.csv", sep=";")
times = pd.read_csv(downloads / "FlightInfo_times.csv")

general = general.drop_duplicates(subset="FlightID")

print("General:", general.shape)
print("Times:", times.shape)

General: (2201, 11)
Times: (2201, 3)


In [8]:
flights = general.merge(times, on="FlightID")

print("Merged:", flights.shape)
flights.head()

Merged: (2201, 13)


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime
0,1,OH,JFK,184,1/1/2004,5935,BWI,0,4,1,N940CA,1455,1455
1,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,1640,1640
2,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,1245,1245
3,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,1815,1709
4,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,1039,1035


In [11]:
flights = general.merge(times, on="FlightID")

print(flights.shape)
flights.head()

(2201, 13)


,FlightID,Carrier,Destination,Distance,Date,FlightNumber,Origin,Weather,DayOfWeek,DayOfMonth,TailNumber,ScheduledDeptTime,ActualDeptTime
0,1,OH,JFK,184,1/1/2004,5935,BWI,0,4,1,N940CA,1455,1455
1,2,DH,JFK,213,1/1/2004,6155,DCA,0,4,1,N405FJ,1640,1640
2,3,DH,LGA,229,1/1/2004,7208,IAD,0,4,1,N695BR,1245,1245
3,4,DH,LGA,229,1/1/2004,7215,IAD,0,4,1,N662BR,1815,1709
4,5,DH,LGA,229,1/1/2004,7792,IAD,0,4,1,N698BR,1039,1035


In [13]:
scheduled = flights["ScheduledDeptTime"].astype(str).str.zfill(4)
actual = flights["ActualDeptTime"].astype(str).str.zfill(4)

flights["ScheduledTime"] = scheduled.str[:2] + ":" + scheduled.str[2:]
flights["ActualTime"] = actual.str[:2] + ":" + actual.str[2:]

flights[["ScheduledDeptTime", "ScheduledTime",
         "ActualDeptTime", "ActualTime"]].head()


,ScheduledDeptTime,ScheduledTime,ActualDeptTime,ActualTime
0,1455,14:55,1455,14:55
1,1640,16:40,1640,16:40
2,1245,12:45,1245,12:45
3,1815,18:15,1709,17:09
4,1039,10:39,1035,10:35


In [16]:
flights["scheduledFull"] = pd.to_datetime(
    flights["Date"] + " " + flights["ScheduledTime"]
)

flights["actualFull"] = pd.to_datetime(
    flights["Date"] + " " + flights["ActualTime"]
)

flights[["Date", "scheduledFull", "actualFull"]].head()

,Date,scheduledFull,actualFull
0,1/1/2004,2004-01-01 14:55:00,2004-01-01 14:55:00
1,1/1/2004,2004-01-01 16:40:00,2004-01-01 16:40:00
2,1/1/2004,2004-01-01 12:45:00,2004-01-01 12:45:00
3,1/1/2004,2004-01-01 18:15:00,2004-01-01 17:09:00
4,1/1/2004,2004-01-01 10:39:00,2004-01-01 10:35:00


In [19]:
overnight = (flights["scheduledFull"] - flights["actualFull"]) > pd.Timedelta(hours=12)

flights.loc[overnight, "actualFull"] = (
    flights.loc[overnight, "actualFull"] + pd.Timedelta(days=1)
)

flights["Delayed"] = (
    (flights["actualFull"] - flights["scheduledFull"]) >= pd.Timedelta(minutes=20)
).astype(int)

flights[["scheduledFull", "actualFull", "Delayed"]].head()

,scheduledFull,actualFull,Delayed
0,2004-01-01 14:55:00,2004-01-01 14:55:00,0
1,2004-01-01 16:40:00,2004-01-01 16:40:00,0
2,2004-01-01 12:45:00,2004-01-01 12:45:00,0
3,2004-01-01 18:15:00,2004-01-01 17:09:00,0
4,2004-01-01 10:39:00,2004-01-01 10:35:00,0


In [24]:
flights["TimeOfDay"] = "Evening"

flights.loc[flights["scheduledFull"].dt.hour < 12, "TimeOfDay"] = "Morning"

flights.loc[
    (flights["scheduledFull"].dt.hour >= 12) &
    (flights["scheduledFull"].dt.hour < 18),
    "TimeOfDay"
] = "Afternoon"

flights[["ScheduledTime", "TimeOfDay"]].head(10)

,ScheduledTime,TimeOfDay
0,14:55,Afternoon
1,16:40,Afternoon
2,12:45,Afternoon
3,18:15,Evening
4,10:39,Morning
5,08:40,Morning
6,12:40,Afternoon
7,16:45,Afternoon
8,17:15,Afternoon
9,21:20,Evening


In [27]:
flights = flights[flights["Carrier"] != "OH"].copy()

print(flights.shape)
print(flights["Carrier"].unique())

(2171, 19)
['DH' 'DL' 'MQ' 'UA' 'US' 'RU' 'CO']


In [29]:
X = flights[
    ["Carrier", "Destination", "Distance", "Origin",
     "Weather", "DayOfWeek", "DayOfMonth", "TimeOfDay"]
].copy()

X["DayOfWeek"] = X["DayOfWeek"].astype(str)

X.head()

,Carrier,Destination,Distance,Origin,Weather,DayOfWeek,DayOfMonth,TimeOfDay
1,DH,JFK,213,DCA,0,4,1,Afternoon
2,DH,LGA,229,IAD,0,4,1,Afternoon
3,DH,LGA,229,IAD,0,4,1,Evening
4,DH,LGA,229,IAD,0,4,1,Morning
5,DH,JFK,228,IAD,0,4,1,Morning


In [30]:
X = pd.get_dummies(
    X,
    columns=["Carrier", "Destination", "Origin", "DayOfWeek", "TimeOfDay"],
    drop_first=True,
    dtype=int
)

print(X.shape)
X.head()

(2171, 21)


,Distance,Weather,DayOfMonth,Carrier_DH,Carrier_DL,Carrier_MQ,Carrier_RU,Carrier_UA,Carrier_US,Destination_JFK,...,Origin_DCA,Origin_IAD,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,DayOfWeek_6,DayOfWeek_7,TimeOfDay_Evening,TimeOfDay_Morning
1,213,0,1,1,0,0,0,0,0,1,...,1,0,0,0,1,0,0,0,0,0
2,229,0,1,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,0
3,229,0,1,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,1,0
4,229,0,1,1,0,0,0,0,0,0,...,0,1,0,0,1,0,0,0,0,1
5,228,0,1,1,0,0,0,0,0,1,...,0,1,0,0,1,0,0,0,0,1


In [32]:
y = flights["Delayed"]

print("X shape:", X.shape)
print("y shape:", y.shape)
print(y.value_counts())

X shape: (2171, 21)
y shape: (2171,)
Delayed
0    1844
1     327
Name: count, dtype: int64


In [33]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.70, random_state=1
)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))

Training rows: 1519
Test rows: 652


In [35]:
from sklearn.neural_network import MLPClassifier

modelNN = MLPClassifier()
modelNN.fit(X_train, y_train)

predicted_test = modelNN.predict(X_test)

print(predicted_test[:10])

[0 0 0 0 0 0 0 0 0 0]


In [38]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

print("Holdout accuracy:", round(accuracy_score(y_test, predicted_test), 3))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, predicted_test, labels=[1, 0]))

print("\nClassification report:")
print(classification_report(y_test, predicted_test))

Holdout accuracy: 0.83

Confusion matrix:
[[ 11 108]
 [  3 530]]

Classification report:
              precision    recall  f1-score   support

           0       0.83      0.99      0.91       533
           1       0.79      0.09      0.17       119

    accuracy                           0.83       652
   macro avg       0.81      0.54      0.54       652
weighted avg       0.82      0.83      0.77       652



In [39]:
from sklearn.model_selection import KFold, cross_val_score

kfold = KFold(n_splits=5, shuffle=True, random_state=1)

cv_accuracy = cross_val_score(MLPClassifier(), X, y, cv=kfold, scoring="accuracy")
cv_precision = cross_val_score(MLPClassifier(), X, y, cv=kfold, scoring="precision")
cv_recall = cross_val_score(MLPClassifier(), X, y, cv=kfold, scoring="recall")

print("5-fold accuracy:", round(cv_accuracy.mean(), 3))
print("5-fold precision:", round(cv_precision.mean(), 3))
print("5-fold recall:", round(cv_recall.mean(), 3))

5-fold accuracy: 0.855
5-fold precision: 0.676
5-fold recall: 0.158


In [42]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.70, random_state=1
)

print(X_train.shape)
print(X_test.shape)

(1519, 21)
(652, 21)
